In [ ]:
# Cell 1: Mount Google Drive (optional, to save models)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torch torchvision torchaudio
!pip install segmentation-models-pytorch
!pip install albumentations
!pip install opencv-python
!pip install tqdm matplotlib

In [ ]:
dataset_path = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset"

In [ ]:
!ls

drive  sample_data


In [ ]:
dataset_path = "/content/Offroad_Segmentation_Training_Dataset"
print(dataset_path)

/content/Offroad_Segmentation_Training_Dataset


In [ ]:
"/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"


'/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset'

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
import albumentations as A
from tqdm import tqdm

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/masks"

VAL_IMAGES   = f"{DATASET_PATH}/val/images"
VAL_MASKS    = f"{DATASET_PATH}/val/masks"

TEST_IMAGES  = f"{DATASET_PATH}/test/images"


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Desert_Dataset"

# Updated paths to reflect the actual unzipped directory structure
TRAIN_IMAGES = f"{DATASET_PATH}/Offroad_Segmentation_Training_Dataset/train/Color_Images"
TRAIN_MASKS  = f"{DATASET_PATH}/Offroad_Segmentation_Training_Dataset/train/Mask_Images"

VAL_IMAGES   = f"{DATASET_PATH}/Offroad_Segmentation_Training_Dataset/val/Color_Images"
VAL_MASKS    = f"{DATASET_PATH}/Offroad_Segmentation_Training_Dataset/val/Mask_Images"

TEST_IMAGES  = f"{DATASET_PATH}/Offroad_Segmentation_Training_Dataset/test/Color_Images"


In [ ]:
IMAGE_SIZE = 256 # Define IMAGE_SIZE

transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE), # Corrected from tResize to Resize
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

In [ ]:
class SegmentationDataset(Dataset):

    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx] # Get the image file name
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name) # Assuming mask file has same name as image

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
        mask = cv2.imread(mask_path, 0) # Read mask as grayscale

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]

        image = torch.tensor(image).permute(2,0,1).float()/255.0
        mask = torch.tensor(mask).long()

        return image, mask



In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/masks"

VAL_IMAGES   = f"{DATASET_PATH}/val/images"
VAL_MASKS    = f"{DATASET_PATH}/val/masks"


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/masks"

VAL_IMAGES   = f"{DATASET_PATH}/val/images"
VAL_MASKS    = f"{DATASET_PATH}/val/masks"


In [ ]:
import torch
import segmentation_models_pytorch as smp

NUM_CLASSES = 2

model = smp.DeepLabV3Plus(
    encoder_name="resnet101",
    encoder_weights="imagenet",
    classes=NUM_CLASSES,
    activation=None
)

# Automatically select GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
print("Using device:", device)


Using device: cpu


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
def train_epoch(loader, model, optimizer):

    model.train()
    total_loss = 0

    for images, masks in tqdm(loader):

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)



In [ ]:
!ls "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset"


ls: cannot access '/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset': No such file or directory


In [ ]:
!ls "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"


ls: cannot access '/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset': No such file or directory


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

BASE_PATH = "/content/drive/MyDrive"

dataset_root = None

# Automatically search dataset folder
for root, dirs, files in os.walk(BASE_PATH):
    if "train" in dirs and "val" in dirs:
        dataset_root = root
        break

print("Detected dataset root:", dataset_root)

if dataset_root is None:
    raise RuntimeError("Dataset folder not found in MyDrive")

# Correct dataset paths
TRAIN_IMAGES = os.path.join(dataset_root, "train", "Color_Images")
TRAIN_MASKS  = os.path.join(dataset_root, "train", "Segmentation")

VAL_IMAGES   = os.path.join(dataset_root, "val", "Color_Images")
VAL_MASKS    = os.path.join(dataset_root, "val", "Segmentation")

TEST_IMAGES  = os.path.join(dataset_root, "test", "Color_Images")

print("\n====== DATASET PATH CHECK ======")

paths = [TRAIN_IMAGES, TRAIN_MASKS, VAL_IMAGES, VAL_MASKS]

for p in paths:
    print(p, "->", "EXISTS" if os.path.exists(p) else "MISSING")


print("\n====== COUNT FILES ======")

train_imgs = sorted(os.listdir(TRAIN_IMAGES))
train_masks = sorted(os.listdir(TRAIN_MASKS))

val_imgs = sorted(os.listdir(VAL_IMAGES))
val_masks = sorted(os.listdir(VAL_MASKS))

print("Train images:", len(train_imgs))
print("Train masks :", len(train_masks))
print("Val images  :", len(val_imgs))
print("Val masks   :", len(val_masks))


Detected dataset root: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset

====== DATASET PATH CHECK ======
/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images -> EXISTS
/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation -> EXISTS
/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Color_Images -> EXISTS
/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation -> EXISTS

====== COUNT FILES ======
Train images: 243
Train masks : 280
Val images  : 258
Val masks   : 280


In [ ]:
!ls /content/drive/MyDrive


'amulya certificate.pdf'
'Amulya Chikati 232576.pdf'
'certportal.eduskillsfoundation.org_Certificate_pdfcrtallllllllllllll.jsp (1)'
'certportal.eduskillsfoundation.org_Certificate_pdfcrtallllllllllllll.jsp (1) (1)'
'Colab Notebooks'
 Desert_Dataset
'detection of rock.pdf'
'Differentiating Functional and Non-Functional Requirements | Gamma'
'Document from Amulya Chikati.pdf'
 eduaxis.zip
 final_masks
 fixed_masks
'GEO TRACK GREEN.pptx'
 hithub1.jpg
 IMG_20250929_115356.jpg
 offroad_deeplabv3_model.pth
 offroad_model_fixed.pth
'Screenshot_2026-03-10-22-11-44-509_com (1).android.chrome.jpg'
'Screenshot_2026-03-10-22-11-44-509_com (2).android.chrome.jpg'
 Screenshot_2026-03-10-22-11-44-509_com.android.chrome.jpg
 Screenshot_2026-03-10-22-16-11-842_com.android.chrome.jpg
 Screenshot_2026-03-10-22-21-25-134_com.android.chrome.jpg
 Screenshot_2026-03-10-22-27-30-759_com.github.android.jpg
 Screenshot_2026-03-10-22-28-01-494_com.github.android.jpg
'Screenshot_2026-03-10-22-28-33-683_com (1).gi

In [ ]:
!ls /content/drive/MyDrive/Offroad_Segmentation_Training_Dataset


ls: cannot access '/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset': No such file or directory


In [ ]:
import os, cv2

print("Train images:", len(os.listdir(TRAIN_IMAGES)))
print("Train masks :", len(os.listdir(TRAIN_MASKS)))

print("Val images  :", len(os.listdir(VAL_IMAGES)))
print("Val masks   :", len(os.listdir(VAL_MASKS)))

# test loading one pair
img_name = os.listdir(TRAIN_IMAGES)[0]

img_path = os.path.join(TRAIN_IMAGES, img_name)
mask_path = os.path.join(TRAIN_MASKS, img_name)

print("Testing image:", img_path)
print("Testing mask :", mask_path)

img = cv2.imread(img_path)
mask = cv2.imread(mask_path)

print("Image shape:", img.shape if img is not None else "ERROR")
print("Mask shape :", mask.shape if mask is not None else "ERROR")

Train images: 243
Train masks : 280
Val images  : 258
Val masks   : 280
Testing image: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images/cc0000012.png
Testing mask : /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation/cc0000012.png
Image shape: (540, 960, 3)
Mask shape : (540, 960, 3)


In [ ]:
# ===== DATASET PATH CONFIG =====

DATASET_PATH = "/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/Segmentation"

VAL_IMAGES   = f"{DATASET_PATH}/val/Color_Images"
VAL_MASKS    = f"{DATASET_PATH}/val/Segmentation"


print("Train images path:", TRAIN_IMAGES)
print("Train masks path:", TRAIN_MASKS)
print("Val images path:", VAL_IMAGES)
print("Val masks path:", VAL_MASKS)

Train images path: /content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images
Train masks path: /content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation
Val images path: /content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/val/Color_Images
Val masks path: /content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation


In [ ]:
mask = cv2.imread(mask_path)
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
mask = torch.tensor(mask, dtype=torch.long)

In [ ]:
mask = cv2.imread(mask_path)

if mask is None:
    raise RuntimeError(f"Mask failed to load: {mask_path}")

mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

mask = torch.tensor(mask, dtype=torch.long)

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/Segmentation"

VAL_IMAGES   = f"{DATASET_PATH}/val/Color_Images"
VAL_MASKS    = f"{DATASET_PATH}/val/Segmentation"


In [ ]:
!ls /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset


Offroad_Segmentation_Training_Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

# =========================
# CORRECT DATASET PATH
# =========================

DATASET_PATH = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = os.path.join(DATASET_PATH, "train", "Color_Images")
TRAIN_MASKS  = os.path.join(DATASET_PATH, "train", "Segmentation")

VAL_IMAGES = os.path.join(DATASET_PATH, "val", "Color_Images")
VAL_MASKS  = os.path.join(DATASET_PATH, "val", "Segmentation")

print("Train images:", TRAIN_IMAGES)
print("Train masks :", TRAIN_MASKS)


# =========================
# DATASET CLASS
# =========================

class OffroadDataset(Dataset):

    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        img_path = os.path.join(self.image_dir, name)
        mask_path = os.path.join(self.mask_dir, name)

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path)

        if image is None:
            raise RuntimeError(f"Image failed: {img_path}")

        if mask is None:
            raise RuntimeError(f"Mask failed: {mask_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        image = torch.tensor(image).permute(2,0,1).float() / 255.0
        mask = torch.tensor(mask, dtype=torch.long)

        return image, mask


# =========================
# CREATE DATASETS
# =========================

train_dataset = OffroadDataset(TRAIN_IMAGES, TRAIN_MASKS)
val_dataset   = OffroadDataset(VAL_IMAGES, VAL_MASKS)


# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)


# =========================
# TEST BATCH
# =========================

images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape :", masks.shape)

print("Dataset loaded successfully.")


Train images: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images
Train masks : /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation
Image batch shape: torch.Size([4, 3, 540, 960])
Mask batch shape : torch.Size([4, 540, 960])
Dataset loaded successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

# =========================
# DATASET PATHS (YOUR PATH)
# =========================

DATASET_PATH = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/Segmentation"

VAL_IMAGES   = f"{DATASET_PATH}/val/Color_Images"
VAL_MASKS    = f"{DATASET_PATH}/val/Segmentation"


print("Train images:", len(os.listdir(TRAIN_IMAGES)))
print("Train masks :", len(os.listdir(TRAIN_MASKS)))
print("Val images  :", len(os.listdir(VAL_IMAGES)))
print("Val masks   :", len(os.listdir(VAL_MASKS)))


# =========================
# HELPER: PAD TO MULTIPLE OF 16
# =========================

def pad_to_16(img):

    h, w = img.shape[:2]

    new_h = ((h + 15) // 16) * 16
    new_w = ((w + 15) // 16) * 16

    pad_h = new_h - h
    pad_w = new_w - w

    img = cv2.copyMakeBorder(
        img,
        0, pad_h,
        0, pad_w,
        cv2.BORDER_CONSTANT,
        value=0
    )

    return img


# =========================
# DATASET CLASS
# =========================

class OffroadDataset(Dataset):

    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path)

        if image is None:
            raise RuntimeError(f"Image failed: {img_path}")

        if mask is None:
            raise RuntimeError(f"Mask failed: {mask_path}")

        # BGR → RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # mask → grayscale
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        # PAD so size divisible by 16
        image = pad_to_16(image)
        mask = pad_to_16(mask)

        # convert to tensor
        image = torch.tensor(image).permute(2,0,1).float() / 255.0
        mask = torch.tensor(mask, dtype=torch.long)

        return image, mask


# =========================
# DATASETS
# =========================

train_dataset = OffroadDataset(TRAIN_IMAGES, TRAIN_MASKS)
val_dataset   = OffroadDataset(VAL_IMAGES, VAL_MASKS)


# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)


# =========================
# TEST BATCH
# =========================

images, masks = next(iter(train_loader))

print("Image batch:", images.shape)
print("Mask batch :", masks.shape)

print("\nDataset ready. Input size is now divisible by 16.")


Train images: 243
Train masks : 280
Val images  : 258
Val masks   : 280
Image batch: torch.Size([4, 3, 544, 960])
Mask batch : torch.Size([4, 544, 960])

Dataset ready. Input size is now divisible by 16.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# =========================
# YOUR DATASET PATH
# =========================

DATASET_PATH = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMAGES = f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASKS  = f"{DATASET_PATH}/train/Segmentation"

VAL_IMAGES   = f"{DATASET_PATH}/val/Color_Images"
VAL_MASKS    = f"{DATASET_PATH}/val/Segmentation"


# =========================
# MASK CONVERSION
# =========================

def convert_mask(mask):
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    mask = mask // 64
    mask[mask > 3] = 3
    return mask.astype("int64")


# =========================
# DATASET CLASS
# =========================

class OffroadDataset(Dataset):

    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        img = cv2.imread(os.path.join(self.img_dir, name))
        mask = cv2.imread(os.path.join(self.mask_dir, name))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = convert_mask(mask)

        img = cv2.resize(img, (512,512))
        mask = cv2.resize(mask, (512,512), interpolation=cv2.INTER_NEAREST)

        img = torch.tensor(img).permute(2,0,1).float()/255
        mask = torch.tensor(mask).long()

        return img, mask


# =========================
# DATASETS
# =========================

train_dataset = OffroadDataset(TRAIN_IMAGES, TRAIN_MASKS)
val_dataset   = OffroadDataset(VAL_IMAGES, VAL_MASKS)


# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    drop_last=True
)


# =========================
# MODEL
# =========================

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    classes=4,
    activation=None
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# =========================
# TRAIN FUNCTION
# =========================

def train_epoch(loader):

    model.train()
    total_loss = 0

    for imgs, masks in tqdm(loader):

        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        preds = model(imgs)

        loss = criterion(preds, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# =========================
# VALIDATION FUNCTION
# =========================

def val_epoch(loader):

    model.eval()
    total_loss = 0

    with torch.no_grad():

        for imgs, masks in loader:

            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)

            loss = criterion(preds, masks)

            total_loss += loss.item()

    return total_loss / len(loader)


# =========================
# TRAINING
# =========================

EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss = train_epoch(train_loader)
    val_loss = val_epoch(val_loader)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)


# =========================
# SAVE MODEL
# =========================

torch.save(model.state_dict(), "/content/drive/MyDrive/offroad_deeplabv3_model.pth")
print("Model saved to Drive")


device: cpu


100%|██████████| 121/121 [23:01<00:00, 11.42s/it]



Epoch 1/5
Train Loss: 0.5841073928293118
Val Loss: 0.2454368232987648


100%|██████████| 121/121 [22:17<00:00, 11.05s/it]



Epoch 2/5
Train Loss: 0.12115202862615428
Val Loss: 0.10041025985581006


...


Epoch 3/5
Train Loss: 0.056617549166452785
Val Loss: 0.05273384967631148


100%|██████████| 121/121 [22:41<00:00, 11.25s/it]



Epoch 4/5
Train Loss: 0.034118386801363025
Val Loss: 0.03406723285483759


100%|██████████| 121/121 [22:12<00:00, 11.02s/it]



Epoch 5/5
Train Loss: 0.022537981460163416
Val Loss: 0.021984652579985848
Model saved to Drive


In [ ]:
torch.save(model.state_dict(), "segmentation_model.pth")


In [ ]:
def predict(image_path):

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img,(512,512))
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0).float()/255

    img = img.to(device)

    with torch.no_grad():
        pred = model(img)

    pred = torch.argmax(pred,1).cpu().numpy()[0]

    return pred

In [ ]:
import os
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_PATH="/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMG=f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASK=f"{DATASET_PATH}/train/Segmentation"

VAL_IMG=f"{DATASET_PATH}/val/Color_Images"
VAL_MASK=f"{DATASET_PATH}/val/Segmentation"


# -------- MASK CONVERSION --------
def convert_mask(mask):

    mask=cv2.cvtColor(mask,cv2.COLOR_BGR2RGB)

    label=np.zeros((mask.shape[0],mask.shape[1]),dtype=np.uint8)

    label[(mask==[255,0,0]).all(axis=2)]=1
    label[(mask==[0,255,0]).all(axis=2)]=2
    label[(mask==[0,0,255]).all(axis=2)]=3

    return label

In [ ]:
import os
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# =========================
# YOUR DATASET PATH
# =========================

DATASET_PATH = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMG  = f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASK = f"{DATASET_PATH}/train/Segmentation"

VAL_IMG  = f"{DATASET_PATH}/val/Color_Images"
VAL_MASK = f"{DATASET_PATH}/val/Segmentation"


# =========================
# MASK CONVERSION
# =========================

def convert_mask(mask):

    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    label = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.uint8)

    label[(mask == [255,0,0]).all(axis=2)] = 1
    label[(mask == [0,255,0]).all(axis=2)] = 2
    label[(mask == [0,0,255]).all(axis=2)] = 3

    return label


# =========================
# DATASET CLASS
# =========================

class OffroadDataset(Dataset):

    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(img_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        img = cv2.imread(os.path.join(self.img_dir, name))
        mask = cv2.imread(os.path.join(self.mask_dir, name))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = convert_mask(mask)

        img = cv2.resize(img, (512,512))
        mask = cv2.resize(mask, (512,512), interpolation=cv2.INTER_NEAREST)

        img = torch.tensor(img).permute(2,0,1).float()/255
        mask = torch.tensor(mask).long()

        return img, mask


# =========================
# DATASETS
# =========================

train_dataset = OffroadDataset(TRAIN_IMG, TRAIN_MASK)
val_dataset   = OffroadDataset(VAL_IMG, VAL_MASK)


# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, drop_last=True)


# =========================
# MODEL
# =========================

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    classes=4,
    activation=None
).to(device)


loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# =========================
# TRAIN FUNCTION
# =========================

def train_epoch(loader):

    model.train()
    total = 0

    for imgs, masks in tqdm(loader):

        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        pred = model(imgs)

        loss = loss_fn(pred, masks)

        loss.backward()
        optimizer.step()

        total += loss.item()

    return total / len(loader)


# =========================
# TRAINING
# =========================

EPOCHS = 5

for e in range(EPOCHS):

    loss = train_epoch(train_loader)

    print("Epoch", e+1, "Loss:", loss)


# =========================
# SAVE MODEL
# =========================

torch.save(model.state_dict(), "/content/drive/MyDrive/offroad_model_fixed.pth")

print("Model saved to Drive")


device: cpu


100%|██████████| 60/60 [24:23<00:00, 24.38s/it]


Epoch 1 Loss: 0.8213121657570203


100%|██████████| 60/60 [22:40<00:00, 22.68s/it]


Epoch 2 Loss: 0.24486681694785753


...

Epoch 3 Loss: 0.1267882070193688


100%|██████████| 60/60 [22:40<00:00, 22.67s/it]


Epoch 4 Loss: 0.08213642127811908


100%|██████████| 60/60 [22:51<00:00, 22.86s/it]


Epoch 5 Loss: 0.05884666641553243
Model saved to Drive


In [ ]:
print("Unique classes predicted:", np.unique(mask))

Unique classes predicted: [ 0  1  2  3 27 39]


In [ ]:
import os
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# YOUR DATASET PATH
# =========================

DATASET_PATH = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

TRAIN_IMG = f"{DATASET_PATH}/train/Color_Images"
TRAIN_MASK = f"{DATASET_PATH}/train/Segmentation"

VAL_IMG = f"{DATASET_PATH}/val/Color_Images"
VAL_MASK = f"{DATASET_PATH}/val/Segmentation"


# =========================
# MASK CONVERSION
# =========================

def convert_mask(mask):

    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    label = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.uint8)

    label[(mask == [255,0,0]).all(axis=2)] = 1
    label[(mask == [0,255,0]).all(axis=2)] = 2
    label[(mask == [0,0,255]).all(axis=2)] = 3

    return label


# =========================
# DATASET CLASS
# =========================

class OffroadDataset(Dataset):

    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(img_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        img = cv2.imread(os.path.join(self.img_dir, name))
        mask = cv2.imread(os.path.join(self.mask_dir, name))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = convert_mask(mask)

        img = cv2.resize(img, (512,512))
        mask = cv2.resize(mask, (512,512), interpolation=cv2.INTER_NEAREST)

        img = torch.tensor(img).permute(2,0,1).float() / 255
        mask = torch.tensor(mask).long()

        return img, mask


# =========================
# DATASETS
# =========================

train_dataset = OffroadDataset(TRAIN_IMG, TRAIN_MASK)
val_dataset   = OffroadDataset(VAL_IMG, VAL_MASK)


# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, drop_last=True)


# =========================
# MODEL
# =========================

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    classes=4,
    activation=None
).to(device)


# =========================
# LOSS & OPTIMIZER
# =========================

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)


# =========================
# TRAIN FUNCTION
# =========================

def train_epoch(loader):

    model.train()
    total_loss = 0

    for imgs, masks in tqdm(loader):

        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        preds = model(imgs)

        loss = loss_fn(preds, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# =========================
# TRAINING
# =========================

EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss = train_epoch(train_loader)

    print("Epoch", epoch+1, "Loss:", train_loss)


# =========================
# SAVE MODEL
# =========================

torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/offroad_model_fixed.pth"
)

print("Training finished and model saved.")


Device: cpu


100%|██████████| 60/60 [23:14<00:00, 23.24s/it]


Epoch 1 Loss: 0.6970722317695618


100%|██████████| 60/60 [23:02<00:00, 23.04s/it]


Epoch 2 Loss: 0.18715470855434735


...

Epoch 3 Loss: 0.10532600271205107


100%|██████████| 60/60 [22:51<00:00, 22.86s/it]


Epoch 4 Loss: 0.06961692105978727


100%|██████████| 60/60 [23:06<00:00, 23.10s/it]


Epoch 5 Loss: 0.05000059859206279
Training finished and model saved.


In [ ]:
import torch
import numpy as np

def compute_iou(pred, mask, num_classes):
    ious = []

    pred = pred.view(-1)
    mask = mask.view(-1)

    for cls in range(num_classes):
        pred_inds = pred == cls
        target_inds = mask == cls

        intersection = (pred_inds[target_inds]).long().sum().item()
        union = pred_inds.long().sum().item() + target_inds.long().sum().item() - intersection

        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)

    return np.nanmean(ious)

In [ ]:
model.eval()

ious = []

with torch.no_grad():
    for images, masks in val_loader:

        images = images.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu()

        iou = compute_iou(preds, masks, num_classes=4)

        ious.append(iou)

# dataset size
print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

# mean IoU
print("Mean IoU:", np.mean(ious))


Train dataset size: 243
Validation dataset size: 258
Mean IoU: 1.0


In [ ]:
print(len(train_dataset))
print(len(val_dataset))


243
258


In [ ]:
images = images.to(device)
outputs = model(images)
preds = torch.argmax(outputs, dim=1)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
from PIL import Image

# Correct mask folder from your output
mask_folder = "/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed"

# Output folder
output_folder = "/content/drive/MyDrive/final_masks"

os.makedirs(output_folder, exist_ok=True)

files = os.listdir(mask_folder)

for file in files:

    path = os.path.join(mask_folder, file)

    mask = np.array(Image.open(path))

    # convert RGB → single channel
    if len(mask.shape) == 3:
        mask = mask[:,:,0]

    # convert to binary labels
    mask = (mask > 128).astype(np.uint8)

    Image.fromarray(mask).save(os.path.join(output_folder, file))

print("Masks fixed successfully")


Masks fixed successfully


In [ ]:
import os
import numpy as np
from PIL import Image
import random

mask_path = "/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed"

files = os.listdir(mask_path)

print("Total mask files:", len(files))

if len(files) == 0:
    print("Mask folder is empty. Check the correct path.")
else:
    file = random.choice(files)

    mask = np.array(Image.open(os.path.join(mask_path, file)))

    print("Selected file:", file)
    print("Unique labels:", np.unique(mask))


Total mask files: 0
Mask folder is empty. Check the correct path.


In [ ]:
import os

root = "/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset"

for path, dirs, files in os.walk(root):
    print(path, " -> ", len(files), "files")


/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset  ->  0 files
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train  ->  0 files
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed  ->  0 files
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset  ->  0 files
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train  ->  0 files
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed  ->  0 files


In [ ]:
import os

root = "/content/drive/MyDrive/Desert_Dataset"

for path, dirs, files in os.walk(root):
    print(path)
    for d in dirs:
        print("   📁", d)
    for f in files[:3]:
        print("   🖼️", f)


/content/drive/MyDrive/Desert_Dataset
   📁 Offroad_Segmentation_Training_Dataset
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset
   📁 train
   📁 Offroad_Segmentation_Training_Dataset
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train
   📁 Segmentation_fixed
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset
   📁 train
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train
   📁 Segmentation_fixed
/content/drive/MyDrive/Desert_Dataset/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed


In [ ]:
import os

dataset_root = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset"

train_img_dir = os.path.join(dataset_root, "train", "Color_Images")
train_mask_dir = os.path.join(dataset_root, "train", "Segmentation_fixed")

print("Dataset root:", dataset_root)
print()

print("Train Image Folder Exists:", os.path.exists(train_img_dir))
print("Train Mask Folder Exists:", os.path.exists(train_mask_dir))
print()

if os.path.exists(train_img_dir):
    images = os.listdir(train_img_dir)
    print("Total Train Images:", len(images))
    print("Sample Images:", images[:5])

print()

if os.path.exists(train_mask_dir):
    masks = os.listdir(train_mask_dir)
    print("Total Train Masks:", len(masks))
    print("Sample Masks:", masks[:5])


Dataset root: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset

Train Image Folder Exists: False
Train Mask Folder Exists: False




In [ ]:
import os
import numpy as np
from PIL import Image

# Define the correct dataset root based on previous os.walk output
dataset_root = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

# Define paths for original and fixed mask directories
train_original_masks_dir = os.path.join(dataset_root, "train", "Segmentation")
train_fixed_masks_dir = os.path.join(dataset_root, "train", "Segmentation_fixed")

val_original_masks_dir = os.path.join(dataset_root, "val", "Segmentation")
val_fixed_masks_dir = os.path.join(dataset_root, "val", "Segmentation_fixed")

mask_processing_pairs = [
    (train_original_masks_dir, train_fixed_masks_dir),
    (val_original_masks_dir, val_fixed_masks_dir)
]

print("Starting mask preprocessing...")

for original_mask_folder, fixed_mask_folder in mask_processing_pairs:
    if not os.path.exists(original_mask_folder):
        print(f"Warning: Original mask folder not found: {original_mask_folder}. Skipping.")
        continue

    os.makedirs(fixed_mask_folder, exist_ok=True)
    print(f"Processing masks in: {original_mask_folder}")
    print(f"Saving fixed masks to: {fixed_mask_folder}")

    files = os.listdir(original_mask_folder)

    for file in files:
        path = os.path.join(original_mask_folder, file)
        mask = np.array(Image.open(path))

        # Convert RGB mask to single channel if it's 3-channel
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]

        # Convert to binary labels (0 background, 1 object) assuming labels are 0 or > 128
        mask = (mask > 128).astype(np.uint8)

        Image.fromarray(mask).save(os.path.join(fixed_mask_folder, file))

    print(f"Fixed masks saved for: {original_mask_folder}")

print("All masks processed successfully. You can now re-run the training code.")


Starting mask preprocessing...
Processing masks in: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation
Saving fixed masks to: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed
Fixed masks saved for: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation
Processing masks in: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation
Saving fixed masks to: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation_fixed
Fixed masks saved for: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation
All masks processed successfully. You can now re-run the training code.


In [ ]:
import os

dataset_root = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset"

train_img_dir = None
train_mask_dir = None

for root, dirs, files in os.walk(dataset_root):

    if "Color_Images" in root:
        train_img_dir = root

    if "Segmentation_fixed" in root:
        train_mask_dir = root

print("Detected Image Folder:", train_img_dir)
print("Detected Mask Folder:", train_mask_dir)


Detected Image Folder: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Color_Images
Detected Mask Folder: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation_fixed


In [ ]:
import os

dataset_root = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset"

train_img_dir = None
train_mask_dir = None

for root, dirs, files in os.walk(dataset_root):

    if "Color_Images" in root:
        train_img_dir = root

    if "Segmentation_fixed" in root:
        train_mask_dir = root

print("Detected Image Folder:", train_img_dir)
print("Detected Mask Folder:", train_mask_dir)


Detected Image Folder: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Color_Images
Detected Mask Folder: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation_fixed


In [ ]:
!ls "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset"


Offroad_Segmentation_Training_Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [51]:
import os

# The dataset_root variable is already defined in the kernel state.
# Let's inspect its content.
print(f"Inspecting contents of: {dataset_root}")

if os.path.exists(dataset_root):
    print("-----------------------------------------")
    print(f"Contents of {dataset_root}:")
    for item in sorted(os.listdir(dataset_root)):
        path = os.path.join(dataset_root, item)
        if os.path.isdir(path):
            print(f"  📁 {item}/")
            # List subdirectories within train/val to find Color_Images and Segmentation_fixed
            if item in ['train', 'val']:
                for sub_item in sorted(os.listdir(path)):
                    sub_path = os.path.join(path, sub_item)
                    if os.path.isdir(sub_path):
                        print(f"    📁 {sub_item}/")
        else:
            print(f"  📄 {item}")
    print("-----------------------------------------")
    print("Please examine the output above carefully.")
    print("Identify the correct paths for your 'Color_Images' and 'Segmentation_fixed' folders.")
    print("Then, update the `train_img_dir` and `train_mask_dir` variables in cell `g_XkpBO4Mu7U` with these precise paths and re-run that cell.")
else:
    print(f"Error: The dataset root path does not exist: {dataset_root}")
    print("Please check your Google Drive structure.")

Inspecting contents of: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset
Error: The dataset root path does not exist: /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset
Please check your Google Drive structure.


In [52]:
import os

print("Listing contents of /content/drive/MyDrive. This may take a moment if you have many files.")
print("--------------------------------------------------------------------------------------")

# Recursively list all directories and up to 5 files in each directory
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    print(f"Directory: {root}")

    if dirs:
        print("  Subdirectories:")
        for d in dirs[:5]: # Limit to first 5 for brevity
            print(f"    - {d}/")

    if files:
        print("  Files (first 5):")
        for f in files[:5]: # Limit to first 5 for brevity
            print(f"    - {f}")
    print("\n") # Add a newline for better readability between directories

print("--------------------------------------------------------------------------------------")
print("Please examine the output above very carefully.")
print("Look for a directory that contains your 'Offroad_Segmentation_Training_Dataset' (or a similar name).")
print("Once you find the correct path (e.g., /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset), please provide it.")

Listing contents of /content/drive/MyDrive. This may take a moment if you have many files.
--------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------
Please examine the output above very carefully.
Look for a directory that contains your 'Offroad_Segmentation_Training_Dataset' (or a similar name).
Once you find the correct path (e.g., /content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset), please provide it.


In [1]:

!pip install huggingface_hub

In [2]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
login()

In [4]:
notebook_path = "/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/desert_segmentation_deeplabv3.ipynb"


In [8]:
from huggingface_hub import login
login()


In [10]:
from huggingface_hub import login
login()


In [11]:
from huggingface_hub import create_repo

create_repo(
    repo_id="amulyachikati/desert-segmentation-deeplabv3",
    repo_type="dataset",
    exist_ok=True
)


RepoUrl('https://huggingface.co/datasets/amulyachikati/desert-segmentation-deeplabv3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='amulyachikati/desert-segmentation-deeplabv3')

In [12]:
import os
print(os.path.exists("/content/drive/MyDrive/spark/desert_segmentation_deeplabv3.ipynb"))


False


In [19]:
import os

base = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base):
    for file in files:
        if "desert" in file.lower():
            print(os.path.join(root, file))


In [20]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "desert_segmentation" in file:
            print(os.path.join(root, file))


In [22]:
import os
print(os.path.exists("/content/drive/MyDrive/spark/desert_segmentation_deeplabv3.ipynb"))


False


In [23]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "desert" in file.lower():
            print(os.path.join(root, file))


In [25]:
import os
print(os.path.exists("/content/drive/MyDrive/Colab Notebooks/desert_segmentation_deeplabv3.ipynb"
))


False


In [27]:
import torch


In [29]:
import torch

# check if model exists
print("Model exists:", "model" in globals())


Model exists: False


In [30]:
import torch
from torchvision.models.segmentation import deeplabv3_resnet50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = deeplabv3_resnet50(weights=None, num_classes=2)
model = model.to(device)

print("Model recreated")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 46.0MB/s]


Model recreated


In [46]:
import os

found_img_dir = None
found_mask_dir = None

print("Searching for dataset directories...")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "Color_Images" in dirs and found_img_dir is None:
        found_img_dir = os.path.join(root, "Color_Images")
    if "Segmentation_fixed" in dirs and found_mask_dir is None:
        found_mask_dir = os.path.join(root, "Segmentation_fixed")

    if found_img_dir and found_mask_dir:
        break # Stop searching once both are found

print("-----------------------------------------")
print(f"Found Image Directory: {found_img_dir}")
print(f"Found Mask Directory: {found_mask_dir}")
print("-----------------------------------------")

if found_img_dir and found_mask_dir:
    print("Please update the `img_dir` and `mask_dir` variables in the `7Dc996EyajFO` cell with these paths and re-run that cell, and then re-run the training cell `g_XkpBO4Mu7U`.")
else:
    print("Could not find one or both directories. Please ensure they exist in your Google Drive.")

Searching for dataset directories...
-----------------------------------------
Found Image Directory: None
Found Mask Directory: None
-----------------------------------------
Could not find one or both directories. Please ensure they exist in your Google Drive.


In [47]:
import os

print("Images exist:", os.path.exists("/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images"
))
print("Masks exist:", os.path.exists("/content/drive/MyDrive/spark/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation_fixed"))


Images exist: False
Masks exist: False


In [48]:
import os

print("Listing contents of /content/drive/MyDrive. This may take a moment if you have many files.")
print("--------------------------------------------------------------------------------------")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    # Print the current directory being explored
    print(f"Directory: {root}")

    # Print up to 5 subdirectories
    if dirs:
        print("  Subdirectories:")
        for d in dirs[:5]: # Limit to first 5 for brevity
            print(f"    - {d}/")

    # Print up to 5 files
    if files:
        print("  Files (first 5):")
        for f in files[:5]: # Limit to first 5 for brevity
            print(f"    - {f}")
    print("\n") # Add a newline for better readability between directories

print("--------------------------------------------------------------------------------------")
print("Please examine the output above to find the exact paths for your 'Color_Images' and 'Segmentation_fixed' folders.")
print("Once found, update the `img_dir` and `mask_dir` variables in cell `7Dc996EyajFO` with these correct paths, then re-run cell `7Dc996EyajFO` and subsequently the training cell `g_XkpBO4Mu7U`.")

Listing contents of /content/drive/MyDrive. This may take a moment if you have many files.
--------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------
Please examine the output above to find the exact paths for your 'Color_Images' and 'Segmentation_fixed' folders.
Once found, update the `img_dir` and `mask_dir` variables in cell `7Dc996EyajFO` with these correct paths, then re-run cell `7Dc996EyajFO` and subsequently the training cell `g_XkpBO4Mu7U`.
